In [1]:
import os
from dataclasses import dataclass

from common import configure_notebook


@dataclass(frozen=True)
class OpenRouterConfig:
    model: str
    base_url: str
    api_key: str
    env_file: str

    def summary(self) -> str:
        masked_key = "loaded" if self.api_key else "missing"
        return (
            f"env file : {self.env_file}\n"
            f"base url : {self.base_url}\n"
            f"model    : {self.model}\n"
            f"api key  : {masked_key}"
        )


env_path = configure_notebook()

config = OpenRouterConfig(
    model="moonshotai/kimi-k2.6",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY", ""),
    env_file=env_path.name,
)

assert config.api_key, "Add OPENROUTER_API_KEY to .env first."
assert not config.model.startswith("replace-"), "Set the model in this cell."
print(config.summary())

env file : .env
base url : https://openrouter.ai/api/v1
model    : moonshotai/kimi-k2.6
api key  : loaded


## Tool/Function Callling

1. The application sends the conversation and tool definitions to the model.
2. The model requests a tool with specific inputs.
3. The application checks the request and runs the tool.
4. The tool’s result is sent back to the model.
5. The model uses the result to answer or request another tool.

In [2]:
# Tool 1
from urllib.parse import quote, urlencode
from urllib.request import urlopen


def get_weather(city: str, units: str = "metric"):
    """Return current weather for a named city. Raise on service failure."""
    if not isinstance(city, str) or not city.strip():
        raise ValueError("city must be nonempty text")
    units = "metric" if units is None else units
    if units not in {"metric", "imperial"}:
        raise ValueError("units must be metric or imperial")
    # %t is temperature. m/u selects Celsius/Fahrenheit; %f is feels-like.
    query = urlencode({"format": "%C %t", "m" if units == "metric" else "u": ""})
    url = f"https://wttr.in/{quote(city.strip(), safe='')}?{query}"
    with urlopen(url, timeout=10) as response:
        weather = response.read().decode("utf-8").strip()
    if not weather:
        raise ValueError("Weather service returned an empty result")
    return f"The weather in {city} is {weather}."


sample_city = "Greater Noida"
get_weather(sample_city, units="metric")

'The weather in Greater Noida is Haze +28°C.'

In [3]:
# Tool 2
import subprocess

ACTIONS = {
    "list_directory": ["ls", "-la"],
    "disk_usage": ["df", "-h"],
    "current_time": ["date"],
    "uptime": ["uptime"],
}


def run_system_action(action: str):
    """Run one predefined, read-only host command. Raise on failure."""
    if not isinstance(action, str) or action not in ACTIONS:
        raise ValueError(f"Unknown system action: {action!r}")
    result = subprocess.run(ACTIONS[action], capture_output=True, text=True, timeout=5, check=True)
    return result.stdout.strip() or "Command ran with no output."


sample_action = "current_time"
print(run_system_action(sample_action))

Wed Sep  9 02:16:37 IST 2026


In [4]:
# Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Retrieves current weather conditions and temperature for a named city using the wttr.in public weather service. Use this when the user asks about current weather or temperature in a specific, real city. Do not use this for forecasts spanning multiple days, historical weather data, or vague locations that are not a resolvable city name.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city to check, e.g. 'Paris', 'Nairobi', 'Osaka'. Must be a real city name, not coordinates or a region."
                },
                "units": {
                    "type": ["string", "null"],
                    "enum": ["metric", "imperial", None],
                    "description": "Temperature unit system. 'metric' returns Celsius, 'imperial' returns Fahrenheit. If null, defaults to metric."
                }
            },
            "required": ["city", "units"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "run_system_action",
        "description": "Runs one predefined, read-only diagnostic command on the host and returns its output. Use this when the user asks for basic system diagnostics such as disk usage, directory contents, current time, or uptime. Never use this to modify, delete, install, or configure anything, and never pass raw shell text; only the fixed set of actions in the enum is permitted.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": ["list_directory", "disk_usage", "current_time", "uptime"],
                    "description": "The specific diagnostic action to run. Each value maps to exactly one fixed, non-destructive command."
                }
            },
            "required": ["action"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [6]:
import json
from openai import OpenAI

client = OpenAI(
    api_key=config.api_key,
    base_url=config.base_url,
)
response = client.responses.create(
    model=config.model,
    instructions="You are a helpful assistant.",
    input="Weather in Greater Noida",
    tools=tools,
)

from common import inspect_response
temp = inspect_response(response)


════════════════════════════════════════════════════════════════════════
                       RESPONSES API · INSPECTION                       
════════════════════════════════════════════════════════════════════════

STATE
────────────────────────────────────────────────────────────────────────
  status                    completed
  model                     moonshotai/kimi-k2.6
  server duration           1.000 s

INPUT
────────────────────────────────────────────────────────────────────────
  request input             not supplied; pass request_input=the_input_used_for_create

OUTPUT · ASSISTANT
────────────────────────────────────────────────────────────────────────
(No assistant text returned.)

INTERMEDIATE RESULTS · REASONING
────────────────────────────────────────────────────────────────────────
  returned reasoning        The user is asking for the weather in Greater Noida. I need to use the get_weather function with city "Greater Noida" and units "metric" (default). Let 

In [9]:
FUNCTION_MAP = {
    "get_weather": get_weather,
    "run_system_action": run_system_action,
}

def execute_tool_call(item):
    args = json.loads(item.arguments)
    result = FUNCTION_MAP[item.name](**args)
    return {
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": str(result),
    }

def run_conversation(user_message: str):
      conversation = [
          {
              "role": "user",
              "content": user_message,
          }
      ]

      while True:
          response = client.responses.create(
              model=config.model,
              instructions="You are a helpful assistant.",
              input=conversation,
              tools=tools,
          )
          inspect_response(response)

          tool_calls = [
              item
              for item in response.output
              if item.type == "function_call"
          ]

          if not tool_calls:
              return response.output_text

          # Preserve the model's function calls in the conversation.
          conversation.extend(response.output)

          # Add the results corresponding to those calls.
          conversation.extend(
              execute_tool_call(item)
              for item in tool_calls
          )

print(run_conversation("What's the weather in Greater Noida? and also help me to understand what is the current time"))


════════════════════════════════════════════════════════════════════════
                       RESPONSES API · INSPECTION                       
════════════════════════════════════════════════════════════════════════

STATE
────────────────────────────────────────────────────────────────────────
  status                    completed
  model                     moonshotai/kimi-k2.6
  server duration           5.000 s

INPUT
────────────────────────────────────────────────────────────────────────
  request input             not supplied; pass request_input=the_input_used_for_create

OUTPUT · ASSISTANT
────────────────────────────────────────────────────────────────────────
(No assistant text returned.)

INTERMEDIATE RESULTS · REASONING
────────────────────────────────────────────────────────────────────────
  returned reasoning        The user is asking for two things:
                            1. The weather in Greater Noida
                            2. The current time
         